# Quick Start

This notebook walks through the core FactoriaX API: creating an environment, resetting it, stepping through episodes, and running batched rollouts on GPU with `jax.vmap`.

**Prerequisites**: install the package with `uv sync` (or `pip install factoriax`).

## Check if GPU is available

This code checks if the GPU is available. If it is not available for any reason, check [the JAX documentation for instructions](https://docs.jax.dev/en/latest/installation.html) on how to install JAX for the GPU.

In [ ]:
import jax

devices = jax.devices()
print("JAX devices:", devices)
if any(d.platform == "gpu" for d in devices):
    print("GPU is available.")
else:
    print("No GPU found — running on CPU. See the JAX installation docs to enable GPU support.")


## Create an environment

`factoriax.make` mirrors `gymnax.make`: pass a registered scenario id and receive an `(env, params)` pair.
Available scenarios: `"EasyRocket-v1"` (16×16), `"Rocket-v1"` (32×32).

In [ ]:
import jax
import jax.numpy as jnp
import factoriax
from factoriax.make import env_from_name

env, params = env_from_name("EasyRocket-v1")
print("observation space:", env.observation_space(params))
print("num actions:      ", factoriax.engine.constants.NUM_ITEM_TYPES)

## Reset and inspect state

In [ ]:
key = jax.random.PRNGKey(0)
key, reset_key = jax.random.split(key)

obs, state = env.reset_env(reset_key, params)
print("obs shape:", obs.shape)
print("step:     ", state.timestep)

## Visualize a reset state

`JaxRenderer` converts an `EnvState` to an RGBA pixel array using pure JAX gathers. Tt runs on GPU and can be vmapped over batched states. `jit_render_map` JIT-compiles the render pass on first call.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from factoriax.engine.jax_renderer import JaxRenderer

renderer = JaxRenderer(tile_px=16)
img = renderer.jit_render_map(state)  # (H, W, 4) uint8 RGBA

plt.figure(figsize=(5, 5))
plt.imshow(np.array(img))
plt.axis("off")
plt.show()


### Map with inventory panel

`render_inventory_panel` renders the player's item counts as an RGB panel. Placing it beside the map gives a combined view useful for debugging rollouts.


In [ ]:
from factoriax.analysis.inventory import render_inventory_panel

map_img = np.array(img)                           # (H, W, 4) RGBA
inventory = np.array(state.player_inventory[0])   # counts indexed by ItemType

inv_panel = render_inventory_panel(inventory, width=320, height=map_img.shape[0])

fig, axes = plt.subplots(
    1, 2, figsize=(10, 4),
    gridspec_kw={"width_ratios": [map_img.shape[1], inv_panel.shape[1]]},
)
axes[0].imshow(map_img)
axes[0].axis("off")
axes[0].set_title("Map")
axes[1].imshow(inv_panel)
axes[1].axis("off")
axes[1].set_title("Inventory")
plt.tight_layout()
plt.show()


## Step with a random action

In [ ]:
from factoriax.engine.actions import Action

key, step_key, action_key = jax.random.split(key, 3)
action = jax.random.randint(action_key, shape=(), minval=0, maxval=len(Action))

obs, state, reward, done, info = env.step_env(step_key, state, action, params)
print("reward:", reward)
print("done:  ", done)
print("step:  ", state.timestep)

## Run a short episode

Loop for a couple of steps and collect rewards. This is slow because you are not running the environments in parallel yet.

In [ ]:
key, reset_key = jax.random.split(key)
obs, state = env.reset_env(reset_key, params)
n_steps = 10

rewards = []
for _ in range(n_steps):
    key, step_key, action_key = jax.random.split(key, 3)
    action = jax.random.randint(action_key, shape=(), minval=0, maxval=len(Action))
    obs, state, reward, done, info = env.step_env(step_key, state, action, params)
    rewards.append(float(reward))

print(f"total reward over {n_steps} steps: {sum(rewards):.3f}")

## Batched rollouts with `jax.vmap`

FactoriaX state is pure JAX arrays, so `vmap` over a batch of environments runs the full batch in a single XLA kernel, without a Python loop. A GPU is necessary for this to be most efficient.

In [ ]:
N_ENVS = 64 # Number of parallel envs

vmap_reset = jax.vmap(env.reset_env, in_axes=(0, None))
vmap_step  = jax.vmap(env.step_env,  in_axes=(0, 0, 0, None))

keys = jax.random.split(jax.random.PRNGKey(1), N_ENVS)
obs_batch, state_batch = vmap_reset(keys, params)
print("batched obs shape:", obs_batch.shape)  # (64, obs_dim)

In [ ]:
# One batched step
step_keys   = jax.random.split(jax.random.PRNGKey(2), N_ENVS)
action_keys = jax.random.split(jax.random.PRNGKey(3), N_ENVS)
actions = jax.vmap(lambda k: jax.random.randint(k, shape=(), minval=0, maxval=len(Action)))(action_keys)

obs_batch, state_batch, rewards, dones, _ = vmap_step(step_keys, state_batch, actions, params)
print("rewards shape:", rewards.shape)   # (64,)
print("mean reward:  ", rewards.mean())

## Next steps

- {doc}`../api/index` — full API reference.